In [1]:
# imports
import os, sys, time

# import the __functions.py (custom functions)
sys.path.append(os.getcwd()) # add code folder to system path
from __functions import *  # imports all custom functions

# local data directories
datadir = '/Users/mcc/Library/CloudStorage/Box-Box/MCC/data/'
projdir = os.path.dirname(os.getcwd())
print(projdir)

print("Complete!")

/Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation
Complete!


In [ ]:
# read in the incidents data (gathered from the ICS209+ by Evan Herpe et al)
fp = os.path.join(projdir, "data/tabular/incidents/ics209plus-wf_incidents_2014_2022.csv")
incis = pd.read_csv(fp)
incis.head()

In [ ]:
if int(incis['MTBS_ID'].isna().sum()) > 0:
    # Just keep fires with MTBS IDs for now, join to perimeter data
    incis = incis[~incis['MTBS_ID'].isnull()]
    len(incis)
else:
    print(f"All fires have MTBS IDs (N = {len(incis)})")

In [ ]:
# load the MTBS perimeter data
fp = os.path.join(datadir, "wildfire/MTBS/mtbs_perimeter_data/mtbs_perims_DD.shp") # stored locally
mtbs = gpd.read_file(fp)
mtbs = mtbs[['Event_ID','geometry']]

# join incidents to MTBS
inci_mtbs = incis.merge(mtbs, on="Event_ID")
inci_mtbs = gpd.GeoDataFrame(
    inci_mtbs,
    geometry=inci_mtbs["geometry"],  # or use the correct column name for MTBS geometry
    crs=mtbs.crs  # use the CRS from the original MTBS GeoDataFrame
)

# reproject to WGS
inci_mtbs = inci_mtbs.to_crs("EPSG:4326")
print(f"Matched [{len(inci_mtbs)}/{len(incis)}]")
del mtbs

# save this file out.
out_fp = os.path.join(projdir, 'data/spatial/wf_incidents_2014_2022_mtbs_perims.gpkg')
inci_mtbs.to_file(out_fp)
print(f"Saved to: {out_fp}")

In [ ]:
# Load counties and join to get list of FIPS
counties = os.path.join(datadir,"boundaries/political/TIGER/tl_2024_us_county/tl_2024_us_county.shp")
counties = gpd.read_file(counties).to_crs("EPSG:4326")

# Intersect with fires
fire_fips = gpd.sjoin(counties, inci_mtbs, how="inner", predicate="intersects")
unique_fips = fire_fips["GEOID"].unique()
print(f"{len(unique_fips)} counties intersect with fire perimeters.\n")

# create a lookup table for fires
# Group by Fire ID and collect FIPS codes (GEOID)
fire_fips_lookup = (
    fire_fips.groupby("Event_ID")["GEOID"]
    .unique()  # get unique FIPS per fire
    .reset_index()
)
# save a table for later
out_fp = os.path.join(projdir,"data/tabular/fire_to_fips.csv")
fire_fips_lookup.to_csv(out_fp, index=False)
print(fire_fips.head())

In [ ]:
# Download the NSI data for all FIPS
# see __functions.py
nsi_results, failed_fips = fetch_nsi_fips(unique_fips, timeout=120)

# Retry on failed downloads
max_retries = 3
retry_count = 0
while failed_fips and retry_count < max_retries:
    print(f"\nRetrying {len(failed_fips)} failed FIPS (Attempt {retry_count + 1})")
    time.sleep(5)  # Optional small pause between rounds
    retry_results, failed_fips = fetch_nsi_fips(failed_fips, timeout=120)
    nsi_results.extend(retry_results)
    retry_count += 1

if failed_fips:
    print(f"Final failed FIPS after {max_retries} retries: {failed_fips}")
    # Optional: save to file
    with open("failed_fips.txt", "w") as f:
        for fips in failed_fips:
            f.write(f"{fips}\n")
else:
    print("All FIPS successfully downloaded.")

In [ ]:
# Turn into a dict for quick lookup
fire_to_fips = dict(zip(fire_fips_lookup["Event_ID"], fire_fips_lookup["GEOID"]))

In [ ]:
# concatenate the results into geodataframe for each fire
nsi = gpd.GeoDataFrame(pd.concat(nsi_results, ignore_index=True), geometry="geometry", crs="EPSG:4326")
print(f"Retrieved {len(nsi)} NSI structure points.")

# extract within fire perimeter
inci_mtbs = inci_mtbs.to_crs("EPSG:4326")
nsi_fire = gpd.sjoin(nsi, inci_mtbs[["MTBS_ID", "Incid_Name", "Ig_Date", "geometry"]], how="inner", predicate="within")
print(f"{len(nsi_fire)} NSI structures fall within fire perimeters.")
print(f"\n{nsi_fire.columns}\n")

# Save outputs
# nsi.to_file(os.path.join(projdir,"data/spatial/nsi_structures.geojson"), driver="GeoJSON")
out_fp = os.path.join(projdir,"data/spatial/nsi_structures_in_fires.geojson")
nsi_fire.to_file(out_fp, driver="GeoJSON")
print(f"Save structure data to: {out_fp}")

In [ ]:
# # save a larger dataset within a buffer of fire perimeters
# buffer = inci_mtbs.copy()
# buffer['geometry'] = inci_mtbs.to_crs('EPSG:5070').geometry.buffer(10000) # make sure were in meters for the buffer
# buffer = buffer.to_crs('EPSG:4326') # and make to WGS for joining to NSI
# # perform the spatial join
# nsi_fire10k = gpd.sjoin(nsi, buffer[["MTBS_ID", "Incid_Name", "Ig_Date", "geometry"]], how="inner", predicate="within")
# print(f"{len(nsi_fire10k)} NSI structures are within 10km of fire perimeters.")
# print(f"\n{nsi_fire10k.columns}\n")
#
# # Save outputs
# out_fp = os.path.join(projdir,"data/spatial/nsi_structures_in_fires_50km.geojson")
# nsi_fire10k.to_file(out_fp, driver="GeoJSON")
# print(f"Save structure data to: {out_fp}")